# Proyecto FIFA World Cup 2022
## Parte 1: puntos 1 al 4

Archivo separado para trabajo y commit del primer integrante. Contiene la presentación general, librerías, parámetros y los puntos 1 a 4 del pipeline.

# ¿Qué es un pipeline de preprocesamiento?

Un pipeline de preprocesamiento es una secuencia ordenada de pasos que transforma los datos originales en un conjunto de datos limpio, consistente y listo para el análisis.

En este notebook el flujo será:

**Obtener → Explorar → Limpiar → Transformar → Escalar → Validar**

Cada etapa se implementa mediante funciones reutilizables. Esta decisión permite separar responsabilidades, evitar duplicación de código y facilitar la trazabilidad del proceso. Además, ayuda a que cualquier integrante del equipo pueda ejecutar nuevamente el notebook y obtener los mismos resultados.

# Evidencia del flujo operacional del pipeline

Para cumplir la arquitectura solicitada en la rúbrica, el notebook mantiene una relación directa entre entrada, procesamiento y salida:

```text
Dataset FIFA World Cup 2022 en data/raw
        ↓
cargar_datos(ruta)
        ↓
explorar_dataframe(df)
        ↓
normalizar_nombres_columnas(df)
        ↓
convertir_porcentaje(df, columna)
        ↓
transformar_fecha_hora(df)
        ↓
crear_variables_resultado(df)
        ↓
codificar_one_hot_auto(df, columna, prefijo)
        ↓
escalar_caracteristicas(df, columnas)
        ↓
validar_dataset(df, grupos_one_hot)
        ↓
Dataset final limpio, numérico, validado y listo para análisis posteriores
```

Este esquema evidencia la coherencia entre las entradas, el procesamiento y las salidas del pipeline de Fase 2.


# Introducción

La Copa Mundial FIFA Qatar 2022 generó una gran cantidad de información estadística relacionada con el desempeño de las selecciones participantes. El dataset utilizado contiene información por partido, incluyendo posesión de balón, goles, intentos, pases, tarjetas, faltas, corners, offsides y otras variables asociadas al rendimiento deportivo.

La problemática definida en la Fase 1 se orienta a comprender qué estadísticas de juego pueden estar asociadas con la victoria de una selección durante el torneo. Para avanzar hacia ese propósito, en esta Fase 2 es necesario construir un proceso técnico que permita obtener, limpiar, transformar y validar el conjunto de datos.

El objetivo de esta etapa no es todavía construir un modelo predictivo final, sino preparar los datos de manera reproducible y documentada. Esto fortalece la consistencia metodológica del proyecto, porque cada transformación queda explicada, ejecutada y verificada dentro del notebook.

# Objetivo general de la Fase 2

Construir un pipeline reproducible de obtención, limpieza, transformación, escalamiento y validación del dataset **FIFA World Cup 2022**, con el fin de dejar los datos preparados para posteriores análisis estadísticos, visualizaciones y/o modelos asociados al rendimiento de las selecciones.

# Objetivos específicos

1. Cargar el dataset desde un archivo CSV utilizando una función reutilizable y controlada.
2. Explorar la estructura inicial del conjunto de datos, identificando dimensiones, tipos de datos, valores nulos, duplicados y categorías relevantes.
3. Limpiar el dataset verificando valores faltantes, registros duplicados e inconsistencias de formato.
4. Transformar columnas de porcentaje, fecha, hora y variables categóricas para convertirlas a formatos numéricos utilizables.
5. Crear variables derivadas, como diferencia de goles y resultado del equipo 1, para apoyar el análisis posterior.
6. Escalar variables numéricas continuas para dejarlas en una escala comparable.
7. Validar técnicamente el dataset resultante mediante verificaciones, pruebas con `assert`, casos normales y excepciones controladas.

# Dataset utilizado

El archivo utilizado corresponde a **Fifa_world_cup_matches.csv**, con estadísticas de partidos de la Copa Mundial FIFA Qatar 2022.

Entre las variables disponibles se encuentran:

- `team1` y `team2`: selecciones participantes en cada partido.
- `possession team1`, `possession team2` y `possession in contest`: posesión expresada como porcentaje.
- `number of goals team1` y `number of goals team2`: goles anotados por cada equipo.
- `date`, `hour` y `category`: información temporal y fase del torneo.
- Variables de rendimiento: intentos, pases, tarjetas, faltas, offsides, corners, tiros al arco y presiones defensivas.

Estas variables permiten construir un dataset procesado para analizar relaciones entre estadísticas de juego y resultado deportivo.

# Librerías utilizadas

| Librería | Uso principal |
|---|---|
| `numpy` | Cálculo numérico y validaciones. |
| `pandas` | Carga, limpieza y transformación de datos tabulares. |
| `matplotlib` | Visualización de control mediante gráficos. |
| `sklearn.preprocessing` | Codificación One-Hot y escalamiento con StandardScaler. |

Se utiliza `np.random.seed(42)` para fijar la semilla aleatoria del entorno. Aunque este notebook no depende fuertemente de procesos aleatorios, esta práctica fortalece la reproducibilidad exigida en la fase.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

# Reproducibilidad del entorno
np.random.seed(42)

%matplotlib inline

In [ ]:
# Parámetros de control reproducible del pipeline
RUTA_DATASET_ESPERADA = "../data/raw/Fifa_world_cup_matches.csv"
FILAS_ESPERADAS_MIN = 64
COLUMNAS_ESPERADAS_MIN = 80

print("Configuración de control cargada")
print("Filas mínimas esperadas:", FILAS_ESPERADAS_MIN)
print("Columnas mínimas esperadas:", COLUMNAS_ESPERADAS_MIN)


# 1. Obtención de los datos

En esta etapa se carga el archivo CSV y se transforma en un DataFrame de pandas.

La carga se implementa mediante la función `cargar_datos(ruta)`, que recibe como parámetro la ubicación del archivo. El uso de una función permite reutilizar la lógica de lectura y controlar errores comunes, como una ruta incorrecta o un archivo inexistente.

El bloque `try / except` permite entregar un mensaje claro si el archivo no se encuentra, en lugar de mostrar un error poco comprensible para el usuario final.

In [ ]:
def cargar_datos(ruta):
    """
    Carga el conjunto de datos desde un archivo CSV.

    Parametros
    ----------
    ruta : str
        Ruta del archivo CSV de entrada.

    Retorna
    -------
    pd.DataFrame
        DataFrame con los datos cargados.

    Lanza
    -----
    FileNotFoundError
        Si la ruta indicada no existe.
    """
    try:
        df = pd.read_csv(ruta)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"No se encontro el archivo '{ruta}'. "
            "Verifica que el CSV este en la ruta indicada."
        )

    print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df

La siguiente celda carga el dataset. La ruta principal está pensada para una estructura de repositorio con carpeta `data/raw`. Además, se incluye una ruta alternativa para facilitar la ejecución si el CSV se encuentra en la misma carpeta del notebook.

In [ ]:
rutas_posibles = [
    "../data/raw/Fifa_world_cup_matches.csv",
    "../data/raw/Fifa_world_cup_matches 2.csv"
]

ultima_excepcion = None
for ruta in rutas_posibles:
    try:
        df = cargar_datos(ruta)
        print(f"Archivo utilizado: {ruta}")
        break
    except FileNotFoundError as e:
        ultima_excepcion = e
else:
    raise ultima_excepcion

Mostramos las primeras filas para confirmar visualmente que las columnas fueron cargadas correctamente y que el contenido coincide con el dataset esperado.

In [ ]:
df.head()

# 2. Exploración inicial

Antes de modificar el dataset, se realiza una exploración inicial. Esta etapa establece una línea base del estado original de los datos.

La función `explorar_dataframe(df)` reporta:

- Dimensiones del dataset.
- Tipos de datos por columna.
- Valores nulos por variable.
- Estadísticos descriptivos de variables numéricas.

Esta evidencia permite justificar técnicamente las decisiones de limpieza y transformación posteriores.

In [ ]:
def explorar_dataframe(df):
    """
    Genera un resumen exploratorio inicial del DataFrame.

    Parametros
    ----------
    df : pd.DataFrame
        Dataset a explorar.

    Retorna
    -------
    pd.DataFrame
        Estadisticos descriptivos de las variables numericas.
    """
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos de variables numericas:")
    return df.describe()

explorar_dataframe(df)

También se revisan las variables categóricas principales. Esto permite identificar categorías inesperadas, errores de tipeo o valores que deban normalizarse antes de aplicar transformaciones.

In [ ]:
columnas_categoricas_iniciales = ['team1', 'team2', 'date', 'hour', 'category']

for col in columnas_categoricas_iniciales:
    print(f"\n{col}:")
    print(df[col].value_counts().head(15))

# 3. Limpieza de datos

La limpieza busca asegurar que los datos sean completos, consistentes y aptos para procesamiento posterior.

En este dataset se revisan tres aspectos principales:

1. **Valores nulos:** permiten saber si se requiere imputación o eliminación de registros.
2. **Duplicados:** permiten detectar partidos repetidos o registros cargados más de una vez.
3. **Nombres de columnas:** se normalizan para evitar errores por espacios dobles o diferencias de escritura.

A diferencia del ejemplo de accidente cerebrovascular, este dataset FIFA no presenta valores nulos en la carga inicial. Por lo tanto, la limpieza se centra principalmente en estandarizar nombres, revisar duplicados y transformar formatos.

In [ ]:
def normalizar_nombres_columnas(df):
    """
    Normaliza los nombres de columnas eliminando espacios extremos,
    reemplazando espacios multiples por uno solo y convirtiendo a minusculas.

    Esto evita errores por columnas con dobles espacios o diferencias de formato.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

columnas_antes = df.columns.tolist()
df = normalizar_nombres_columnas(df)
columnas_despues = df.columns.tolist()

print("Primeras 10 columnas antes:")
print(columnas_antes[:10])
print("\nPrimeras 10 columnas despues:")
print(columnas_despues[:10])

Luego de normalizar los nombres de columnas, se verifica nuevamente la existencia de valores nulos y duplicados. Esta comprobación deja evidencia directa del estado del dataset antes de las transformaciones.

In [ ]:
print("Total de valores nulos:", int(df.isnull().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))

df.isnull().sum().sort_values(ascending=False).head(10)

Aunque no se detectan valores nulos, se define una función de imputación para mantener una arquitectura robusta y reutilizable. Esta función puede ser usada si en futuras versiones del dataset aparecen valores faltantes en columnas numéricas.

Esto aporta a la rúbrica porque evidencia manejo riguroso de valores NA, aun cuando el dataset actual no requiere imputación efectiva.

In [ ]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa valores nulos en una columna numerica usando media o mediana.

    Parametros
    ----------
    df : pd.DataFrame
        Dataset de entrada.
    columna : str
        Nombre de la columna numerica.
    estrategia : str
        Estrategia de imputacion: 'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        Dataset con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    if not pd.api.types.is_numeric_dtype(df[columna]):
        raise TypeError(f"La columna '{columna}' debe ser numerica para imputar.")

    df = df.copy()
    n_nulos = int(df[columna].isnull().sum())

    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'.")

    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con {estrategia} = {valor:.2f}")
    return df

# 4. Transformación de variables de porcentaje

Las columnas de posesión vienen como texto con el símbolo `%`, por ejemplo `42%`. Para analizarlas estadísticamente o escalarlas, deben convertirse a valores numéricos.

La función `convertir_porcentaje(df, columna)` elimina el símbolo `%` y convierte la columna a tipo `float`. Esta transformación es pertinente porque la posesión de balón es una variable cuantitativa y no debe permanecer como texto.

In [ ]:
def convertir_porcentaje(df, columna):
    """
    Convierte una columna con porcentajes en texto a valores numericos float.

    Ejemplo: '42%' -> 42.0
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    df = df.copy()
    df[columna] = (
        df[columna]
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
        .astype(float)
    )
    return df

columnas_porcentaje = [
    'possession team1',
    'possession team2',
    'possession in contest'
]

for col in columnas_porcentaje:
    df = convertir_porcentaje(df, col)

print(df[columnas_porcentaje].dtypes)
df[columnas_porcentaje].head()

Como control de calidad, se valida que la suma de las tres columnas de posesión sea aproximadamente 100% por partido. Se acepta una diferencia máxima de 1 punto porcentual, porque el dataset original contiene porcentajes redondeados. Esta regla representa una verificación de coherencia interna del dataset.

In [ ]:
suma_posesion = df[columnas_porcentaje].sum(axis=1)
print(suma_posesion.describe())

# Se acepta una diferencia maxima de 1 punto porcentual por redondeo del dataset original.
diferencia_maxima = float((suma_posesion - 100).abs().max())
assert diferencia_maxima <= 1, "La suma de posesion presenta diferencias mayores a 1 punto porcentual."
print(f"[OK] Coherencia de posesion validada. Diferencia maxima por redondeo: {diferencia_maxima}")